Bronze standard notes

- Data DOI: https://doi.org/10.5281/zenodo.17298664
- Model DOI (PyTorch state_dict): https://doi.org/10.5281/zenodo.17298751

This notebook generates the dataset (saved to `../Data/simple_dataset.csv`), trains the PyTorch linear model, and saves its `state_dict` to `Bronze/linear_model.pt`.


In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)
num_rows = 100

age = np.random.randint(18, 65, size=num_rows)
city = np.random.choice(['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix'], size=num_rows)
salary = np.random.randint(45000, 120000, size=num_rows)
user_id = np.arange(1, num_rows + 1)


city_weights = {
    'New York': 15,
    'Los Angeles': 10,
    'Chicago': 5,
    'Houston': 0,
    'Phoenix': -5
}

city_bonus = pd.Series(city).map(city_weights).values

# Define the formula: Score is influenced by age, salary, and city
# We scale salary by dividing by 1000 to keep it from dominating the equation
base_score = (age * 0.5) + (salary / 1000 * 0.3) + city_bonus

noise = np.random.normal(loc=0, scale=5, size=num_rows) # Gaussian noise with mean=0, std=5
final_score = base_score + noise

# Ensure the score stays within a logical range (e.g., 1 to 100) and round it
final_score = np.clip(final_score, 1, 100)
final_score = np.round(final_score, 2)

df = pd.DataFrame({
    'UserID': user_id,
    'Age': age,
    'City': city,
    'Salary': salary,
    'Score': final_score
})

# --- 4. Display and Save ---
print("Generated DataFrame Head (with correlated Score):")
print(df.head())

# Save the DataFrame to a new CSV file
output_path = "../Data/simple_dataset.csv"
df.to_csv(output_path, index=False)

print(f"\nSuccessfully saved dataset with correlated score to '{output_path}'")

Generated DataFrame Head (with correlated Score):
   UserID  Age      City  Salary  Score
0       1   56   Chicago   91717  55.16
1       2   46   Chicago   95859  59.17
2       3   32  New York   71309  51.28
3       4   60   Chicago  108734  71.19
4       5   25   Chicago  115467  54.51

Successfully saved dataset with correlated score to '../Data/simple_dataset.csv'


In [2]:
import pandas as pd

file_path = "../Data/simple_dataset.csv"

df = pd.read_csv(file_path)

print(df.head())

   UserID  Age      City  Salary  Score
0       1   56   Chicago   91717  55.16
1       2   46   Chicago   95859  59.17
2       3   32  New York   71309  51.28
3       4   60   Chicago  108734  71.19
4       5   25   Chicago  115467  54.51


In [7]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)


file_path = "../Data/simple_dataset.csv"

df = pd.read_csv(file_path)

df_processed = pd.get_dummies(df, columns=['City'], drop_first=True)

X = df_processed.drop(['UserID', 'Score'], axis=1).values
y = df_processed['Score'].values.reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=674
)

df_processed = pd.get_dummies(df, columns=['City'], drop_first=True)

# Convert to numpy arrays and ensure proper data types for PyTorch
X_train = X_train.astype(np.float32)
y_train = y_train.astype(np.float32).reshape(-1, 1)
X_test = X_test.astype(np.float32)
y_test = y_test.astype(np.float32).reshape(-1, 1)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

# Create DataLoader objects (optional for simple linear models, but good practice)
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(dataset=train_dataset, batch_size=2, shuffle=True)

# Determine input and output sizes
INPUT_SIZE = X_train.shape[1]
OUTPUT_SIZE = 1 # Score is a single value

class LinearRegressionModel(nn.Module):
    def __init__(self, input_size, output_size):
        super(LinearRegressionModel, self).__init__()
        # nn.Linear performs the operation: y = xA^T + b
        self.linear = nn.Linear(input_size, output_size)

    def forward(self, x):
        # This is the prediction step
        out = self.linear(x)
        return out

# Instantiate the model
pt_model = LinearRegressionModel(INPUT_SIZE, OUTPUT_SIZE)

criterion = nn.MSELoss()
# Adam is a common optimization algorithm
optimizer = optim.Adam(pt_model.parameters(), lr=0.0005)

# 2. Training Loop
num_epochs = 10000
for epoch in range(num_epochs):
    for i, (inputs, targets) in enumerate(train_loader):
        
        # Forward pass: compute predicted y by passing x to the model
        outputs = pt_model(inputs)
        
        # Compute Loss
        loss = criterion(outputs, targets)
        
        # Backward pass (Backpropagation): compute gradient of the loss
        optimizer.zero_grad() # Clear previous gradients
        loss.backward()       # Compute new gradients
        
        # Update Weights: update model parameters
        optimizer.step()
        
    if (epoch + 1) % 20 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')



# 1. Disable gradient calculations for evaluation
pt_model.eval()
with torch.no_grad():
    y_pred_tensor = pt_model(X_test_tensor)

# 2. Convert PyTorch tensors to NumPy for scikit-learn metrics
y_pred_np = y_pred_tensor.numpy()
y_test_np = y_test_tensor.numpy()

# 3. Calculate Performance Metrics
from sklearn.metrics import mean_squared_error, r2_score
mse_pt = mean_squared_error(y_test_np, y_pred_np)
r2_pt = r2_score(y_test_np, y_pred_np)

print("\n--- PyTorch Model Performance ---")
print(f"Mean Squared Error (MSE): {mse_pt:.2f}")
print(f"R-squared (R²): {r2_pt:.2f}")

# 4. Extract and Display Coefficients
# PyTorch coefficients are stored in the linear layer's 'weight' and 'bias' attributes
coefficient_weights = pt_model.linear.weight.data.numpy().flatten()
intercept_bias = pt_model.linear.bias.data.numpy()[0]

feature_names = df_processed.drop(['UserID', 'Score'], axis=1).columns
coeffs_pt = pd.DataFrame(coefficient_weights, feature_names, columns=['Coefficient'])

print("\n--- PyTorch Model Coefficients ---")
print(coeffs_pt)

print("\nIntercept (Bias):")
print(f"{intercept_bias:.4f}")

pt_output_path = 'linear_model.pt'
torch.save(pt_model.state_dict(), pt_output_path)
print(f"\n✅ PyTorch model state_dict saved to: '{pt_output_path}'")

Epoch [20/10000], Loss: 36245452.0000
Epoch [40/10000], Loss: 175416.4688
Epoch [60/10000], Loss: 189.9365
Epoch [80/10000], Loss: 487.9625
Epoch [100/10000], Loss: 11.1254
Epoch [120/10000], Loss: 525.5810
Epoch [140/10000], Loss: 228.5769
Epoch [160/10000], Loss: 40.9111
Epoch [180/10000], Loss: 484.8931
Epoch [200/10000], Loss: 208.0171
Epoch [220/10000], Loss: 169.1624
Epoch [240/10000], Loss: 574.3943
Epoch [260/10000], Loss: 11.5144
Epoch [280/10000], Loss: 18.7862
Epoch [300/10000], Loss: 312.1680
Epoch [320/10000], Loss: 27.7620
Epoch [340/10000], Loss: 173.7210
Epoch [360/10000], Loss: 22.5731
Epoch [380/10000], Loss: 650.8701
Epoch [400/10000], Loss: 84.0899
Epoch [420/10000], Loss: 463.3453
Epoch [440/10000], Loss: 563.1295
Epoch [460/10000], Loss: 372.1385
Epoch [480/10000], Loss: 260.6573
Epoch [500/10000], Loss: 33.8852
Epoch [520/10000], Loss: 40.8961
Epoch [540/10000], Loss: 9.9188
Epoch [560/10000], Loss: 14.0570
Epoch [580/10000], Loss: 79.1110
Epoch [600/10000], Loss